# Generate .nas mesh

In [2]:
# Read in data from swc file of a bifurcation and create vascularmd mesh for it
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pyvista as pv
from ArterialTree import ArterialTree

# ── Configuration ─────────────────────────────────────────────────────────────
data_folder_name = 'BG0014.CNG'
bifurcation_id   = 134
N = 48       # nodes around each cross-section circumference (must be multiple of 8)
d = 0.25     # longitudinal cross-section density
layer_ratio = [0.2, 0.4, 0.4]  # O-grid radius ratio [a, b, c], a+b+c = 1
num_a, num_b = 4, 4            # number of O-grid layers in parts a and b

swc_path = os.path.join('..', 'Data', data_folder_name, f'bifurcation_{bifurcation_id}.swc')
out_dir  = os.path.join('..', 'Output')
os.makedirs(out_dir, exist_ok=True)

# Build the vascularmd (ArterialTree) model straight from the SWC centerline data
tree = ArterialTree("patient", "Aneurisk", swc_path)
tree.model_network()                               # parametric modeling of the network
tree.compute_cross_sections(N, d, parallel=False)   # mesh cross sections

volume_mesh = tree.mesh_volume(layer_ratio, num_a, num_b)
print(f"Meshed full network: {volume_mesh.n_points} points, {volume_mesh.n_cells} cells")

# ── Write the volume mesh to Nastran (.nas), tagging every element as one domain ──
# Native route (no VTK/pyvista hop): get the mesh as a meshio.Mesh built straight
# from the tree's internal arrays, then attach a per-cell property id via
# cell_data["nastran:ref"]. meshio's Nastran writer emits that value into the
# CHEXA PID field (field 3, blank by default). Every hexahedron gets PID 1
# ("fluid"), so the whole mesh reads as one explicitly-tagged solid domain that
# COMSOL recognises as a single selection. Nastran PIDs are integers, so the name
# "fluid" is just our label for PID 1.
mesh = tree.get_volume_mesh("meshio")               # meshio.Mesh(points, [("hexahedron", conn)])
n_cells = mesh.cells[0].data.shape[0]
FLUID_PID = 1
mesh.cell_data["nastran:ref"] = [np.full(n_cells, FLUID_PID, dtype=int)]

nas_path = os.path.join(out_dir, f'bifurcation_{bifurcation_id}_volume_mesh.nas')
mesh.write(nas_path)
print(f"Saved Nastran mesh -> {os.path.abspath(nas_path)}")
print(f"  all {n_cells} CHEXA tagged with PID {FLUID_PID} (= 'fluid'), one domain")


Loading swc file...
Modeling the network...
Meshing bifurcations.
Meshing edges.
Meshing volume...
Meshed full network: 69751 points, 66144 cells
Saved Nastran mesh -> /home/xbfl0349/arterial_network/Output/bifurcation_134_volume_mesh.nas
  all 66144 CHEXA tagged with PID 1 (= 'fluid'), one domain
